# Disambiguating researcher–author candidates
### A dependency-aware walkthrough with synthetic pipeline data

This notebook illustrates what happens after name matching has produced candidate author profiles. All people and publication records are fictitious.

## What does disambiguation do?

Name matching finds author profiles that could belong to a researcher. Disambiguation then evaluates the candidate profiles for each researcher: it compares name evidence and uses available publication, coauthor, citation, subject-area, and affiliation evidence to score profile pairs. It clusters profiles judged likely to represent the same person, keeps distinct profiles separate when evidence suggests different people, and selects or reports the best-supported researcher-level author group.

It does **not** re-run name matching unconditionally, nor can it guarantee that uncertain candidates are correct. Pair decisions, clusters, final selections, and candidate scores are saved separately so the evidence and outcome can be inspected.

## Dependency behavior in this example

The cell below invokes only the `name_disambiguation` stage through its command-line interface. Disambiguation consumes the canonical matcher artifact owned by `[name_matching.outputs]` in the same TOML config. If that artifact is missing, the stage command runs the configured name-matching producer first. If it already exists, it is validated and reused; freshness is not checked, so an existing artifact may be out of date. The notebook does not call the matching stage directly.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

repo_root = Path.cwd()
while not (repo_root / 'pyproject.toml').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').is_file():
    raise RuntimeError('Start Jupyter from within the academic-inbreeding repository.')
sys.path.insert(0, str(repo_root / 'src'))

from acib.constants import AUTHOR_ID, EID, RESEARCHER_ID
from acib.name_disambiguation.config import load_name_disambiguation_config
from acib.name_disambiguation.constants import (
    AUTHOR_IDS,
    CITED_EID,
    CITING_EID,
    CLUSTER_ID,
    CLUSTER_SIZE,
    ARTICLE_PARENT_SUBJECT_OVERLAP_COUNT,
    CONFIDENCE,
    CURRENT_AFFILIATION_OVERLAP_COUNT,
    CURRENT_AFFILIATION_OVERLAP_IDS,
    DECISION_LABEL,
    DISAMBIGUATED_CLUSTER_ID,
    LEFT_AUTHOR_ID,
    PROFILE_PARENT_SUBJECT_OVERLAP_COUNT,
    ARTICLE_PARENT_SUBJECT_OVERLAP_CODES,
    PROFILE_PARENT_SUBJECT_OVERLAP_CODES,
    RAW_SCORE,
    RIGHT_AUTHOR_ID,
    SELECTED_AUTHOR_IDS,
    SELECTION_REASON,
    SELECTION_STATUS,
)
from acib.name_matching.constants import NAME, RESEARCHER_NAME

## 1. Load the shared synthetic configuration

In [2]:
config_path = repo_root / 'config' / 'pipeline_synthetic.toml'
config = load_name_disambiguation_config(config_path)
config.check_external_inputs_exist()

print(f'Config: {config_path.relative_to(repo_root)}')
print(f'Canonical matching dependency: {config.matching_output.relative_to(repo_root)}')
print(f'Matcher artifact exists before this run: {config.matching_output.is_file()}')
print(f'Article-author evidence: {config.article_author_file_path.relative_to(repo_root)}')
print(f'Citation edges: {config.article_reference_file_path.relative_to(repo_root)}')
print(f'Disambiguation output directory: {config.outputs.directory.relative_to(repo_root)}')
print(f'Pair thresholds: possible={config.possible_author_score_threshold}, same={config.same_author_score_threshold}')

Config: config/pipeline_synthetic.toml
Canonical matching dependency: data/output/synthetic/name_matching/resolved_researchers.csv
Matcher artifact exists before this run: True
Article-author evidence: data/test/synthetic/article_authors.csv
Citation edges: data/test/synthetic/article_references.csv
Disambiguation output directory: data/output/synthetic/name_disambiguation
Pair thresholds: possible=3.0, same=6.0


## 2. Preview the publication evidence

These tables are external evidence inputs to disambiguation. The matcher artifact is a generated upstream dependency and is resolved from the matching stage's configured output path—not from a second input-path declaration.

In [3]:
article_authors = pd.read_csv(config.article_author_file_path)
article_authors[[EID, AUTHOR_IDS]].head(8)

,eid,author_ids
0,SYN-PUB-001,"[990000001, 990010001]"
1,SYN-PUB-002,"[990000002, 990010001]"
2,SYN-PUB-003,"[990000003, 990010002]"
3,SYN-PUB-004,"[990000004, 990010003]"
4,SYN-PUB-005,"[990000005, 990005005]"
5,SYN-PUB-006,"[990000006, 990005006]"
6,SYN-PUB-007,"[990000007, 990005007]"
7,SYN-PUB-008,"[990000008, 990005008]"


In [4]:
article_references = pd.read_csv(config.article_reference_file_path)
article_references[[CITING_EID, CITED_EID]].head(8)

,citing_eid,cited_eid
0,SYN-PUB-001,SYN-PUB-004
1,SYN-PUB-002,SYN-PUB-004
2,SYN-PUB-004,SYN-PUB-001
3,SYN-PUB-005,SYN-PUB-006
4,SYN-PUB-006,SYN-PUB-007
5,SYN-PUB-007,SYN-PUB-005
6,SYN-PUB-008,SYN-PUB-009
7,SYN-PUB-009,SYN-PUB-010


## 3. Invoke disambiguation only

This runs `python -m acib.name_disambiguation` with the synthetic config. The CLI owns dependency handling: it starts matching only if the required matcher artifact is absent, then runs disambiguation. `--overwrite` replaces this config's synthetic disambiguation artifacts so the notebook can be rerun. It does not request overwriting an existing matcher artifact.

In [5]:
command = [
    sys.executable,
    '-m',
    'acib.name_disambiguation',
    '--config',
    str(config_path),
    '--overwrite',
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=repo_root, check=True)
print('Disambiguation finished.')

Running: /home/chri/PycharmProjects/academic-inbreeding/.venv/bin/python -m acib.name_disambiguation --config /home/chri/PycharmProjects/academic-inbreeding/config/pipeline_synthetic.toml --overwrite
2026-11-03 09:34:33 [info     ] Prepared author candidates     author_rows=21
2026-11-03 09:34:33 [info     ] Building disambiguation scoring context article_author_rows=21 article_reference_rows=17 name_frequency_rows=0
2026-11-03 09:34:33 [info     ] Built disambiguation scoring context
2026-11-03 09:34:33 [info     ] Partitioned author candidates  ambiguous_author_rows=7 ambiguous_batch_count=3 singleton_author_rows=14 singleton_batch_count=14
2026-11-03 09:34:33 [info     ] Processing ambiguous disambiguation batches batch_count=3 max_workers=1
2026-11-03 09:34:33 [info     ] Processing single-candidate disambiguation batches batch_count=14
2026-11-03 09:34:33 [info     ] Finished disambiguation batches batch_count=17
Disambiguation finished.


## 4. Inspect researcher-level selections

The selection artifact contains one final status and selected author group per processed researcher. Status and reason fields explain whether the selection was straightforward or required evidence-based ranking.

In [6]:
selections = pd.read_csv(config.outputs.final_author_selection)
selection_columns = [
    RESEARCHER_ID,
    RESEARCHER_NAME,
    SELECTION_STATUS,
    SELECTED_AUTHOR_IDS,
    SELECTION_REASON,
]
selections[selection_columns].sort_values(RESEARCHER_ID)

,sc_id,researcher_name,status,selected_author_ids,reason
0,990001,"mustermann, max",assigned_by_combined_asjc,"['990000002', '990000001']",best cluster has a unique strongest assigned b...
3,990002,"do, john",assigned_single_candidate,['990000004'],only one candidate author exists for this sc_i...
4,990004,"lovelace, ada",assigned_single_candidate,['990000005'],only one candidate author exists for this sc_i...
5,990005,"al hassan, noor",assigned_single_candidate,['990000006'],only one candidate author exists for this sc_i...
6,990006,"garcia, zo",assigned_single_candidate,['990000007'],only one candidate author exists for this sc_i...
7,990007,"martin, jean luc",assigned_single_candidate,['990000008'],only one candidate author exists for this sc_i...
8,990008,"nguyen, alice marie",assigned_single_candidate,['990000009'],only one candidate author exists for this sc_i...
9,990009,"oconnell, sami",assigned_single_candidate,['990000010'],only one candidate author exists for this sc_i...
10,990010,"de la cruz, ana maria",assigned_single_candidate,['990000011'],only one candidate author exists for this sc_i...
1,990011,"morgan, casey",assigned_single_cluster,"['990000012', '990000019']",only one cluster remained after disambiguation


## 5. Inspect pair decisions and clusters

Pair decisions are evidence-weighted labels for pairs of candidate profiles. Cluster membership shows which profiles were grouped together. The thresholds and scoring rules are configured by the stage; the notebook only displays the resulting artifacts.

In [7]:
pair_decisions = pd.read_csv(config.outputs.pair_decisions)
pair_columns = [
    RESEARCHER_ID,
    LEFT_AUTHOR_ID,
    RIGHT_AUTHOR_ID,
    DECISION_LABEL,
    RAW_SCORE,
    CONFIDENCE,
]
pair_decisions[pair_columns].sort_values([RESEARCHER_ID, RAW_SCORE], ascending=[True, False])

,sc_id,left_author_id,right_author_id,label,raw_score,confidence
1,990001,990000001,990000002,same_author,28.5,0.999
0,990001,990000001,990000003,distinct,10.0,0.924
2,990001,990000003,990000002,distinct,6.0,0.818
3,990011,990000012,990000019,same_author,15.0,0.977
4,990019,990000020,990000021,distinct,15.0,0.977


In [8]:
clusters = pd.read_csv(config.outputs.clusters)
cluster_columns = [
    RESEARCHER_ID,
    CLUSTER_ID,
    DISAMBIGUATED_CLUSTER_ID,
    CLUSTER_SIZE,
    AUTHOR_ID,
    NAME,
]
clusters[cluster_columns].sort_values([RESEARCHER_ID, CLUSTER_ID, AUTHOR_ID])

,sc_id,cluster_id,disambiguated_cluster_id,cluster_size,author_id,name
1,990001,1,990000001,2,990000001,"Mustermann, Max"
0,990001,1,990000001,2,990000002,"Mustermann, M."
2,990001,2,990000003,1,990000003,"Mustermann, Max"
7,990002,1,990000004,1,990000004,"Doe, John"
8,990004,1,990000005,1,990000005,"Lovelace, Ada"
9,990005,1,990000006,1,990000006,"Al-Hassan, Noor"
10,990006,1,990000007,1,990000007,"García, Zoë"
11,990007,1,990000008,1,990000008,"Martin, Jean-Luc"
12,990008,1,990000009,1,990000009,"Nguyen, Alice M."
13,990009,1,990000010,1,990000010,"O'Connell, Sami"


## 6. Two selection examples, step by step

The score table records evidence per possible cluster. The examples below compare the selected cluster with the alternative so the final status is traceable to the evidence columns, rather than being treated as a black-box answer.

### Case A — Max Mustermann: selected by combined subject-area evidence

Two clusters remain. Neither overlaps the researcher’s current-affiliation mapping, so affiliation does not decide this case. Max has inferred ASJC code `1712`; the selected cluster has parent subject code `1700` in **both** article evidence and Scopus profile evidence. The competing profile has no article- or profile-subject overlap. That unique combined overlap explains `assigned_by_combined_asjc`. The parent-code relationship is why the output can show researcher code `1712` while the cluster code is `1700`.

In [9]:
selection_scores = pd.read_csv(config.outputs.final_author_selection_scores)
score_columns = [
    RESEARCHER_ID,
    AUTHOR_IDS,
    ARTICLE_PARENT_SUBJECT_OVERLAP_COUNT,
    ARTICLE_PARENT_SUBJECT_OVERLAP_CODES,
    PROFILE_PARENT_SUBJECT_OVERLAP_COUNT,
    PROFILE_PARENT_SUBJECT_OVERLAP_CODES,
]
max_scores = selection_scores.loc[
    selection_scores[RESEARCHER_ID].eq(990001), score_columns
]
max_scores.sort_values(ARTICLE_PARENT_SUBJECT_OVERLAP_COUNT, ascending=False)


,sc_id,author_ids,article_parent_subject_overlap_count,article_parent_subject_overlap_codes,profile_parent_subject_overlap_count,profile_parent_subject_overlap_codes
0,990001,"['990000002', '990000001']",1,[1700],1,[1700]
1,990001,['990000003'],0,[],0,[]


In [10]:
selections.loc[
    selections[RESEARCHER_ID].eq(990001),
    [RESEARCHER_ID, SELECTION_STATUS, SELECTED_AUTHOR_IDS, SELECTION_REASON],
]


,sc_id,status,selected_author_ids,reason
0,990001,assigned_by_combined_asjc,"['990000002', '990000001']",best cluster has a unique strongest assigned b...


### Case B — Ari Example: selected by current affiliation

Ari also has two candidate clusters, but this synthetic researcher’s university is `TU Wien`, which maps to Scopus affiliation ID `60018163`. The first cluster has one current-affiliation overlap; the alternative has zero. Both clusters have zero article-subject and profile-subject overlap with Ari’s inferred field. Because the affiliation match is unique, the earlier current-affiliation rule resolves the case as `assigned_by_current_affiliation` before ASJC ranking is considered. The names and profiles are fictitious; this real mapping value is included solely to exercise the existing rule.

In [11]:
ari_scores = selection_scores.loc[
    selection_scores[RESEARCHER_ID].eq(990019), score_columns
]
ari_scores.sort_values(CURRENT_AFFILIATION_OVERLAP_COUNT, ascending=False)


KeyError: 'current_affiliation_overlap_count'

In [ ]:
selections.loc[
    selections[RESEARCHER_ID].eq(990019),
    [RESEARCHER_ID, SELECTION_STATUS, SELECTED_AUTHOR_IDS, SELECTION_REASON],
]


### Why were there two clusters in the first place?

Clustering happens before final selection. In these examples, the same-name profiles have widely separated publication histories and are labeled `distinct`; ASJC or affiliation then helps choose between the resulting clusters. Max’s other pair of profiles is labeled `same_author` and is grouped together.

In [ ]:
example_pairs = pair_decisions.loc[
    pair_decisions[RESEARCHER_ID].isin([990001, 990019]),
    [RESEARCHER_ID, LEFT_AUTHOR_ID, RIGHT_AUTHOR_ID, DECISION_LABEL, RAW_SCORE],
]
example_pairs.sort_values([RESEARCHER_ID, RAW_SCORE], ascending=[True, False])


In [ ]:
max_selection = selections.loc[selections[RESEARCHER_ID].eq(990001)].iloc[0]
ari_selection = selections.loc[selections[RESEARCHER_ID].eq(990019)].iloc[0]
assert max_selection[SELECTION_STATUS] == 'assigned_by_combined_asjc'
assert ari_selection[SELECTION_STATUS] == 'assigned_by_current_affiliation'
assert max_scores[CURRENT_AFFILIATION_OVERLAP_COUNT].max() == 0
assert max_scores[ARTICLE_PARENT_SUBJECT_OVERLAP_COUNT].max() > 0
assert max_scores[PROFILE_PARENT_SUBJECT_OVERLAP_COUNT].max() > 0
assert ari_scores[CURRENT_AFFILIATION_OVERLAP_COUNT].max() == 1
assert ari_scores[CURRENT_AFFILIATION_OVERLAP_COUNT].min() == 0
assert ari_scores[ARTICLE_PARENT_SUBJECT_OVERLAP_COUNT].max() == 0
assert ari_scores[PROFILE_PARENT_SUBJECT_OVERLAP_COUNT].max() == 0
print('Both selection routes and their distinguishing evidence are confirmed.')

## Summary

- Max’s case demonstrates selection through a unique combined article/profile ASJC overlap.
- Ari’s case demonstrates selection through a unique current-affiliation overlap, before subject-area fallback.
- The disambiguation CLI is the only stage explicitly invoked here; it produces missing matching input through the configured dependency mechanism.
- Existing artifacts are checked for presence and contract validity, not freshness.
- Generated files are owned by the configured stage output directories under `data/output/synthetic/`.